# Build Aspect Based Sentiment analysis cho bộ dữ liệu ABSA Tiki book review

In [1]:
# Truy cập drive để lấy dữ liệu
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Tải các thư viện cần thiết
## !pip install transformers
## !pip install torchvision
## !pip install Dataset

## Import mấy cái này nếu cần

!pip install underthesea

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 26.4 MB/s eta 0:00:00


In [3]:
# Import các thư viện cần thiết
import os
os.environ["WANDB_DISABLED"] = "true"
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.utils import resample
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.model_selection import train_test_split
from datasets import Dataset
from underthesea import word_tokenize
print("Libraries imported successfully.")

Libraries imported successfully.


## chạy lần lượt gán nhãn của sentiment đến content, physical, price ...
- Mỗi vòng lặp như vậy sẽ đi gán lại vào bộ dữ liệu tiki

In [4]:
# Load bộ dữ liệu training
df = pd.read_csv('/content/drive/MyDrive/Data Training/train_data_clean.csv')

# Fill các dữ liệu thiếu
if 'content' in df.columns:
    df['content'] = df['content'].fillna("")

display(df.head())

,review_id,rating,review_title,product_name,category,content,content_raw,sentiment_llm,as_content,as_physical,as_price,as_packaging,as_delivery,as_service
0,14357443,4,Hài lòng,Sách Xứ Cát,Sách văn học,"Sản phẩm thật sự chất lượng, và Tiki giao hàng...","Sản phẩm thật sự chất lượng, và Tiki giao hàng...",1,NaN,NaN,1.0,NaN,2.0,NaN
1,16749123,1,Rất không hài lòng,Bộ sách Làm Giàu Từ Chứng Khoán (How To Make M...,Sách kinh tế,không phù hợp đầu tư dài hạn,Ko phù hợp đầu tư dài hạn,0,NaN,NaN,NaN,NaN,NaN,NaN
2,897166,1,Không nên mua,Sách Predictably Irrational : The Hidden Force...,Reference,"Cuốn sách không đúng khổ ghi trên thông tin, k...","Cuốn sách không đúng khổ ghi trên thông tin, k...",0,NaN,2.0,NaN,NaN,NaN,NaN
3,9105241,3,Bình thường,Siêu Cò – How To Be A Power Connector,Sách kinh tế,"Sách hay, nội dung hấp dẫn, thiết thực. Chấm 5...","Sách hay, nội dung hấp dẫn, thiết thực.\nChấm ...",1,2.0,NaN,NaN,NaN,2.0,NaN
4,6820537,4,Hài lòng,Sách Elon Musk: How The Billionaire CEO Of Spa...,Biographies & Memoirs,sách chất lượng in chưa được cao,sách chất lượng in chưa được cao,1,NaN,1.0,NaN,NaN,NaN,NaN


In [5]:
MODEL_NAME = "vinai/phobert-base-v2"
TEXT_COL = "content_raw"
LABEL_COL = "sentiment_llm"   # change to: as_content, as_price, as_delivery, etc.

In [6]:
# Chia tập balanced thành train (80%) và temp (20%)
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df[LABEL_COL],
    random_state=42
)

# Chia tiếp temp thành val (10%) và test (10%)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df[LABEL_COL],
    random_state=42
)

print(f"Kích thước tập train: {len(train_df)}")
print(f"Kích thước tập validation: {len(val_df)}")
print(f"Kích thước tập test: {len(test_df)}")

print("\nPhân bố nhãn trong tập Train:")
display(train_df[LABEL_COL].value_counts())

print("\nPhân bố nhãn trong tập Validation:")
display(val_df[LABEL_COL].value_counts())

print("\nPhân bố nhãn trong tập Test:")
display(test_df[LABEL_COL].value_counts())

Kích thước tập train: 8556
Kích thước tập validation: 1070
Kích thước tập test: 1070

Phân bố nhãn trong tập Train:


,count
sentiment_llm,
0,4486
2,2675
1,1395



Phân bố nhãn trong tập Validation:


,count
sentiment_llm,
0,561
2,335
1,174



Phân bố nhãn trong tập Test:


,count
sentiment_llm,
0,561
2,334
1,175


In [7]:
# Initialize model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3    # 3 classes if labels are 0,1,2
)

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
def vietnamese_word_segmenter(text):
    if not isinstance(text, str) or text.strip() == '':
        return ''
    try:
        # Ensure it returns space-separated words joined by underscores for PhoBERT
        return word_tokenize(text, format='text')
    except Exception as e:
        return str(text)


# 1. Combine content and content_raw
df['combined_content'] = df['content'].astype(str) + " - " + df['content_raw'].astype(str)

# 2. Split the dataset again
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df[LABEL_COL], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df[LABEL_COL], random_state=42)

# 3. Apply word segmentation on the new combined column
train_df['segmented_content'] = train_df['combined_content'].apply(vietnamese_word_segmenter)
val_df['segmented_content'] = val_df['combined_content'].apply(vietnamese_word_segmenter)
test_df['segmented_content'] = test_df['combined_content'].apply(vietnamese_word_segmenter)

# 4. Create Hugging Face Datasets
train_ds = Dataset.from_pandas(train_df[['segmented_content', LABEL_COL]], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[['segmented_content', LABEL_COL]], preserve_index=False)
test_ds = Dataset.from_pandas(test_df[['segmented_content', LABEL_COL]], preserve_index=False)

display(train_df[['combined_content', 'segmented_content']].head())

,combined_content,segmented_content
7414,"Nhỏ hơn bàn tay tí, chưa đọc ,khi nào xong fix...","Nhỏ hơn bàn_tay tí , chưa đọc , khi nào xong f..."
7937,Không bọc bao bì cho cuốn sách - Không bọc bao...,Không bọc bao_bì cho cuốn sách - Không bọc bao...
5186,Sách hay dành cho nhà lãnh đạo. - Sách hay dàn...,Sách hay dành cho nhà_lãnh_đạo . - Sách hay dà...
1509,"Mình thấy sách hay, ý nghĩa cho cả trẻ em và n...","Mình thấy sách hay , ý_nghĩa cho cả trẻ_em và ..."
760,Sách được đóng gói cẩn thận. Lại thêm một cuốn...,Sách được đóng_gói cẩn_thận . Lại thêm một cuố...


In [12]:
#Check dataframe
print("Train_Dataset:")
print(train_ds)

Train_Dataset:
Dataset({
    features: ['segmented_content', 'sentiment_llm'],
    num_rows: 8556
})


In [ ]:
# 6. Re-initialize the base model for a fresh fine-tuning run
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

# 7. Initialize a new Trainer
new_trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_ds,
    eval_dataset=tokenized_val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("Bắt đầu fine-tune với dữ liệu mới (content + content_raw)...")
new_trainer.train()

# 8. Evaluate on the new test set
print("\nĐánh giá trên tập test mới:")
test_results = new_trainer.predict(tokenized_test_ds)
pred_labels = np.argmax(test_results.predictions, axis=1)
true_labels = test_results.label_ids

test_acc = accuracy_score(true_labels, pred_labels)
print(f"Độ chính xác (accuracy) trên tập test: {test_acc:.4f}")
print(classification_report(true_labels, pred_labels, target_names=["Tiêu cực", "Trung lập", "Tích cực"]))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Bắt đầu fine-tune với dữ liệu mới (content + content_raw)...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.709987,0.317338,0.872897,0.822613
2,0.488632,0.314260,0.879439,0.832757
3,0.339309,0.380618,0.882243,0.815655
4,0.216062,0.399652,0.887850,0.851261
5,0.149161,0.502930,0.885047,0.845452
6,0.109363,0.569050,0.885047,0.845725
7,0.059847,0.652185,0.888785,0.844514
8,0.034168,0.682832,0.885981,0.842509
9,0.024102,0.685922,0.896262,0.850448
10,0.018381,0.703321,0.895327,0.852444


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Đánh giá trên tập test mới:


Độ chính xác (accuracy) trên tập test: 0.8869
              precision    recall  f1-score   support

    Tiêu cực       0.92      0.95      0.94       561
   Trung lập       0.69      0.61      0.65       175
    Tích cực       0.91      0.93      0.92       334

    accuracy                           0.89      1070
   macro avg       0.84      0.83      0.83      1070
weighted avg       0.88      0.89      0.88      1070



In [ ]:
train_df['segmented_content'][:10]

,segmented_content
7414,"Nhỏ hơn bàn_tay tí , chưa đọc , khi nào xong f..."
7937,Không bọc bao_bì cho cuốn sách
5186,Sách hay dành cho nhà_lãnh_đạo .
1509,"Mình thấy sách hay , ý_nghĩa cho cả trẻ_em và ..."
760,Sách được đóng_gói cẩn_thận . Lại thêm một cuố...
8940,"Đóng_gói cẩn_thận , nội_dung sách hay . Rất hà..."
7214,( * ) Đánh_giá không tính điểm Khâu đóng_gói k...
879,Quyển sách giúp mình nhìn rõ những vấn_đề tron...
2921,Mình nghĩ với giá này đắt so với chất_lượng sá...
1119,bộ_mặt quyển sách mà thế_này đây : ( Thấy Tiki...


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples['segmented_content'],
        padding='max_length',
        truncation=True,
        max_length=256
    )

# Apply tokenization to all datasets
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Remove the original text column
tokenized_train = tokenized_train.remove_columns(['segmented_content'])
tokenized_val = tokenized_val.remove_columns(['segmented_content'])
tokenized_test = tokenized_test.remove_columns(['segmented_content'])

# Verify
print("Tokenized Train Dataset Features:", tokenized_train.features)
print("\nFirst row of tokenized train dataset:")
print(tokenized_train[0])

Map:   0%|          | 0/8556 [00:00<?, ? examples/s]

Map:   0%|          | 0/1070 [00:00<?, ? examples/s]

Map:   0%|          | 0/1070 [00:00<?, ? examples/s]

Tokenized Train Dataset Features: {'sentiment_llm': Value('int64'), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8'))}

First row of tokenized train dataset:
{'sentiment_llm': 1, 'input_ids': [0, 18552, 48, 3178, 10294, 4, 102, 987, 4, 26, 142, 954, 8520, 2094, 44, 423, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

In [7]:
# các parameter để train
training_args = TrainingArguments(
    output_dir="./sentiment_class",
    do_train=True,
    do_eval=True,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy",
    greater_is_better=True,
    logging_dir="./train_logs",
    logging_steps=100,
    fp16=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

# Tinh compute metric
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

In [ ]:
from transformers import Trainer, EarlyStoppingCallback

# Rename label column to 'labels' as required by the Trainer
train_final = tokenized_train.rename_column(LABEL_COL, "labels")
val_final = tokenized_val.rename_column(LABEL_COL, "labels")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_final,
    eval_dataset=val_final,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("Fine tuning mô hình ...")
trainer.train()

Fine tuning mô hình ...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.283131,0.387517,0.873832,0.835820
2,0.232639,0.483468,0.881308,0.825709
3,0.286192,0.496504,0.881308,0.828789
4,0.182773,0.491340,0.888785,0.842924
5,0.125472,0.547991,0.895327,0.854891
6,0.061974,0.635814,0.895327,0.856185
7,0.026959,0.760092,0.890654,0.838225
8,0.029727,0.722462,0.887850,0.843130


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=2144, training_loss=0.1554795709897333, metrics={'train_runtime': 479.8068, 'train_samples_per_second': 178.322, 'train_steps_per_second': 5.586, 'total_flos': 9004793608323072.0, 'train_loss': 0.1554795709897333, 'epoch': 8.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.5495328307151794,
 'eval_accuracy': 0.8943925233644859,
 'eval_f1_macro': 0.8532302287858847,
 'eval_runtime': 1.9002,
 'eval_samples_per_second': 563.087,
 'eval_steps_per_second': 35.259,
 'epoch': 8.0}

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

# Rename label column in test set to 'labels' so trainer.predict can find the ground truth
tokenized_test_final = tokenized_test.rename_column(LABEL_COL, "labels")

# Đánh giá mô hình trên tập test sử dụng dữ liệu đã chuẩn bị
test_results = trainer.predict(tokenized_test_final)
pred_labels = np.argmax(test_results.predictions, axis=1)
true_labels = test_results.label_ids

# Tính độ chính xác
test_acc = accuracy_score(true_labels, pred_labels)
print(f"Độ chính xác (accuracy) trên tập test: {test_acc:.4f}")

# Báo cáo chi tiết theo từng lớp
class_names = ["Tiêu cực", "Trung lập", "Tích cực"]
print("Báo cáo phân loại trên tập test:")
print(classification_report(true_labels, pred_labels, target_names=class_names))

Độ chính xác (accuracy) trên tập test: 0.8879
Báo cáo phân loại trên tập test:
              precision    recall  f1-score   support

    Tiêu cực       0.95      0.93      0.94       561
   Trung lập       0.66      0.70      0.68       175
    Tích cực       0.92      0.92      0.92       334

    accuracy                           0.89      1070
   macro avg       0.84      0.85      0.84      1070
weighted avg       0.89      0.89      0.89      1070



In [ ]:
# 5. Tokenize the new datasets
tokenized_train_ds = train_ds.map(tokenize_function, batched=True).remove_columns(['segmented_content']).rename_column(LABEL_COL, "labels")
tokenized_val_ds = val_ds.map(tokenize_function, batched=True).remove_columns(['segmented_content']).rename_column(LABEL_COL, "labels")
tokenized_test_ds = test_ds.map(tokenize_function, batched=True).remove_columns(['segmented_content']).rename_column(LABEL_COL, "labels")

print("Tokenization complete for combined data.")

Map:   0%|          | 0/8556 [00:00<?, ? examples/s]

Map:   0%|          | 0/1070 [00:00<?, ? examples/s]

Map:   0%|          | 0/1070 [00:00<?, ? examples/s]

Tokenization complete for combined data.


In [ ]:
from sklearn.utils import resample
import pandas as pd

# Lấy dữ liệu của từng class
df_0 = train_df[train_df[LABEL_COL] == 0]  # Tiêu cực
df_1 = train_df[train_df[LABEL_COL] == 1]  # Trung lập
df_2 = train_df[train_df[LABEL_COL] == 2]  # Tích cực

# Tìm số lượng của class lớn nhất
max_size = max(len(df_0), len(df_1), len(df_2))
print(f"Số lượng mẫu của class lớn nhất (Majority class size): {max_size}")

# Oversample các class ít dữ liệu để bằng với max_size
df_0_up = resample(df_0, replace=True, n_samples=max_size, random_state=42) if len(df_0) < max_size else df_0
df_1_up = resample(df_1, replace=True, n_samples=max_size, random_state=42) if len(df_1) < max_size else df_1
df_2_up = resample(df_2, replace=True, n_samples=max_size, random_state=42) if len(df_2) < max_size else df_2

# Gộp lại thành tập train mới đã được cân bằng
train_df_upsampled = pd.concat([df_0_up, df_1_up, df_2_up])

# Xáo trộn dữ liệu (shuffle) để tránh mô hình học theo thứ tự
train_df_upsampled = train_df_upsampled.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nPhân bố nhãn sau khi Oversampling:")
display(train_df_upsampled[LABEL_COL].value_counts())

Số lượng mẫu của class lớn nhất (Majority class size): 4486

Phân bố nhãn sau khi Oversampling:


,count
sentiment_llm,
0,4486
2,4486
1,4486


In [ ]:
# 1. Tạo lại Dataset và Tokenize cho tập train mới
train_ds_upsampled = Dataset.from_pandas(train_df_upsampled[['segmented_content', LABEL_COL]], preserve_index=False)
tokenized_train_upsampled = train_ds_upsampled.map(tokenize_function, batched=True).remove_columns(['segmented_content']).rename_column(LABEL_COL, "labels")

# 2. Khởi tạo lại model base
oversampled_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

# 3. Dùng Trainer tiêu chuẩn (vì dữ liệu đã cân bằng, không cần class weights)
oversampled_trainer = Trainer(
    model=oversampled_model,
    args=training_args,
    train_dataset=tokenized_train_upsampled,
    eval_dataset=tokenized_val_ds,  # Dùng lại tập val cũ
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("Bắt đầu fine-tune với dữ liệu đã được Oversample...")
oversampled_trainer.train()

# 4. Đánh giá trên tập test
print("\nĐánh giá mô hình Oversampling trên tập test:")
test_results_os = oversampled_trainer.predict(tokenized_test_ds)
pred_labels_os = np.argmax(test_results_os.predictions, axis=1)
true_labels_os = test_results_os.label_ids

acc_os = accuracy_score(true_labels_os, pred_labels_os)
print(f"Độ chính xác (accuracy) trên tập test: {acc_os:.4f}")
print(classification_report(true_labels_os, pred_labels_os, target_names=["Tiêu cực", "Trung lập", "Tích cực"]))

Map:   0%|          | 0/13458 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Bắt đầu fine-tune với dữ liệu đã được Oversample...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.549484,0.327020,0.883178,0.842403
2,0.295145,0.471622,0.880374,0.819386
3,0.161953,0.561567,0.887850,0.834215
4,0.084945,0.604198,0.892523,0.843633
5,0.073405,0.694714,0.886916,0.843617
6,0.030473,0.731153,0.895327,0.852110
7,0.029969,0.753306,0.895327,0.848276
8,0.011152,0.766179,0.897196,0.853040
9,0.024026,0.791596,0.894393,0.853138
10,0.009691,0.800570,0.894393,0.851045


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Đánh giá mô hình Oversampling trên tập test:


Độ chính xác (accuracy) trên tập test: 0.8850
              precision    recall  f1-score   support

    Tiêu cực       0.93      0.95      0.94       561
   Trung lập       0.69      0.60      0.64       175
    Tích cực       0.90      0.93      0.92       334

    accuracy                           0.89      1070
   macro avg       0.84      0.83      0.83      1070
weighted avg       0.88      0.89      0.88      1070

